# LangChain + LangGraph: the one pattern to learn

This notebook exists to kill the confusion from repeated mistakes:
- passing a **class** instead of an **instance** (`StrOutputParser` vs `StrOutputParser()`)
- putting `{placeholder}` text inside a raw `HumanMessage` and expecting it to fill in
- not knowing when to use a plain LCEL chain vs a LangGraph node

There is really only **one core pattern**: `prompt | llm | parser`. Everything else
(tools, branching, memory) is built on top of that. Learn this pattern once, reuse it everywhere.

## Setup

In [2]:
from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)

## Part 1 — The plain LCEL chain: `prompt | llm | parser`

Three rules that fix every mistake you've hit so far:

1. **Build the prompt with `ChatPromptTemplate.from_messages([...])`**, using
   `("system", "...")` / `("human", "...")` tuples. Placeholders like `{question}`
   inside these tuple strings ARE filled in when you `.invoke()`.
   (A raw `HumanMessage(content="{question}")` is NOT a template — it's a literal
   string, `{question}` stays as-is.)
2. **Always instantiate the parser**: `StrOutputParser()`, not `StrOutputParser`.
   The bare class, when piped, gets *called* on the LLM output at runtime and
   crashes with a `BaseModel.__init__()` error.
3. **Chain everything with `|`** and call `.invoke(dict)` with keys matching your
   placeholders. Don't set `output_parser=` inside `ChatPromptTemplate(...)` —
   that field is legacy and ignored; parsing only happens via the `|` chain.

In [3]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

chain = prompt_template | llm | StrOutputParser()

result = chain.invoke({"question": "What is the capital of France?"})
print(result)

The capital of France is Paris.


## Part 2 — Multiple variables

Same pattern, just more placeholders. The keys in the dict you pass to `.invoke()`
must match every `{placeholder}` used across your messages.

In [8]:
prompt_template2 = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate to {language}."),
    ("human", "{text}")
])

chain2 = prompt_template2 | llm | StrOutputParser()

result2 = chain2.invoke({"language": "French", "text": "Good morning"})
print(result2)

System: You are a translator. Translate to Spanish.
Human: Hello
Bonjour.


## Part 3 — Wrapping the SAME chain in a simple linear LangGraph

This is the part that trips people up: a LangGraph node is just a plain Python
function that takes `state` (a dict) and returns a partial dict update. Inside
that function, you use the *exact same* `chain.invoke(...)` from Part 1 — nothing
about the chain changes just because it now lives inside a graph.

Graph shape here is the simplest possible: `START -> answer_node -> END`.

In [5]:
class ChainState(TypedDict):
    question: str
    answer: str


def answer_node(state: ChainState) -> ChainState:
    answer = chain.invoke({"question": state["question"]})
    return {"answer": answer}


graph_builder = StateGraph(ChainState)
graph_builder.add_node("answer_node", answer_node)
graph_builder.add_edge(START, "answer_node")
graph_builder.add_edge("answer_node", END)

graph = graph_builder.compile()

result3 = graph.invoke({"question": "What is the capital of Japan?"})
print(result3)

{'question': 'What is the capital of Japan?', 'answer': 'The capital of Japan is Tokyo.'}


## Part 4 — The messages-based linear graph (used by your chatbot notebooks)

Your other notebooks (e.g. `4_chatbot_without_branching_hitl.ipynb`) use a
`messages` list as state instead of separate `question`/`answer` fields. This is
the standard chat pattern:

- `Annotated[list[BaseMessage], add_messages]` — `add_messages` is a reducer that
  **appends** new messages to the list instead of overwriting it.
- The node calls `llm.invoke(state["messages"])` directly (no prompt template
  needed here since you're just passing the running conversation), and returns
  `{"messages": [response]}` — a list with just the *new* message, which
  `add_messages` merges in.

This is still the same linear shape: `START -> chat_node -> END`.

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


chat_graph_builder = StateGraph(ChatState)
chat_graph_builder.add_node("chat_node", chat_node)
chat_graph_builder.add_edge(START, "chat_node")
chat_graph_builder.add_edge("chat_node", END)

chat_graph = chat_graph_builder.compile()

result4 = chat_graph.invoke({"messages": [HumanMessage(content="Say hi in one short sentence.")]})
print(result4["messages"][-1].content)

Hello.


## Cheat sheet — recap of the fixes

| Mistake | Fix |
|---|---|
| `StrOutputParser` (class) | `StrOutputParser()` (instance) |
| `HumanMessage(content="{question}")` inside a template | `("human", "{question}")` tuple via `ChatPromptTemplate.from_messages` |
| `itemgetter` (bare) | `itemgetter("question")` (called with the key) |
| `output_parser=...` kwarg on `ChatPromptTemplate` | Pipe it: `prompt \| llm \| parser` |
| Confusing "chain" vs "graph" | A graph node is just a function; call your normal chain *inside* it |

**One pattern, two flavors:**
- Standalone logic → `chain = prompt | llm | parser`, then `chain.invoke({...})`
- Needs state/branching/memory/tools → put that same `chain.invoke(...)` (or
  `llm.invoke(...)`) inside a node function, wire nodes with `StateGraph`.